# Modulus Arithmetic with Integer Algebras

**Part II · Geometric Algebra & Core** — Tutorial 09

This tutorial shows pytanga's **integer-valued algebras** with **modular
arithmetic**. Integer dtypes store coefficients exactly, and a modulus turns every
operation into arithmetic over a centred residue ring — the foundation for
lattice-based cryptography and exact symbolic computation.

By the end you will be able to:

- Create an `Algebra` with `dtype="int64"` and a stored `modulus`.
- Explain the **`hmod`** half-space modular reduction and its centred interval.
- Rely on automatic reduction for a single-modulus algebra.
- Use the explicit per-call `_mod` methods when **two moduli** must coexist on one
  algebra (the NTRU pattern).
- Compute modular inverses with `inv()` / `inv(p)`.
- Solve modular GA equations with **`solve_mod`**.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (algebra construction,
> dtypes, products, inverse). The solver internals are covered in
> [Tutorial 11](../11_equation_solving/).

## 1. Setup

Only the core `Algebra` type is needed here. `solve_mod` is imported from
`pytanga.solver.solve` and used in Section 8.

In [1]:
from pytanga import Algebra
from pytanga.solver.solve import solve_mod

## 2. Integer algebras — dtype and modulus

An `Algebra` accepts the same four dtypes as everywhere in pytanga: `float64`
(default), `float32`, `int64`, and `int32`. Integer dtypes store coefficients
**exactly** — no floating-point rounding — and can additionally carry a
**modulus**.

When `modulus` is set at construction, every arithmetic operator (`+`, `-`, `*`,
and scalar multiplication) automatically reduces each coefficient after the
operation, so results stay inside the centred interval of the ring. The modulus
requires an integer dtype.

In [2]:
alg = Algebra(3, 0, dtype="int64", modulus=101)   # G(3,0) over Z/101Z

print("dim        :", alg.dim)
print("sig        :", alg.sig)
print("dtype      :", alg.dtype)
print("modulus    :", alg.modulus)
print("algebra_dim:", alg.algebra_dim, "  (2^3 = 8 basis blades)")

dim        : 3
sig        : 0
dtype      : int64
modulus    : 101
algebra_dim: 8   (2^3 = 8 basis blades)


## 3. Half-space modular reduction — `hmod`

Standard modular reduction maps an integer into `[0, p)`. pytanga instead uses the
**half-space** (centred) variant `hmod`, which maps into the centred interval

    [-(p-1)//2,  (p-1)//2]     (for odd p)

so coefficients stay small in magnitude and wrap symmetrically around zero. For a
value already in `[0, p)` the rule is:

    hmod(v, p) = v        if v <= p // 2
    hmod(v, p) = v - p    otherwise

The C++ engine applies this to every coefficient after every operation. There is
no public Python function named `hmod`; it is reached through `mv.reduce(p)` and
through the fixed-modulus algebra. The one-liner below reproduces it exactly and is
a useful mental model.

In [3]:
def hmod(v: int, p: int) -> int:
    """Half-space modular reduction, matching pytanga's C++ `hmod`."""
    r = v % p
    if r > p // 2:
        r -= p
    return r


# Odd primes give a symmetric centred interval: Z/7Z -> [-3, 3].
for v in (-4, -1, 0, 1, 3, 4, 7, 8):
    print(f"hmod({v:3d}, 7) = {hmod(v, 7):3d}")

print()

# Coefficients exceeding p//2 wrap negative: Z/101Z -> [-50, 50].
for v in (50, 51, 60, 100, 101):
    print(f"hmod({v:3d}, 101) = {hmod(v, 101):3d}")

hmod( -4, 7) =   3
hmod( -1, 7) =  -1
hmod(  0, 7) =   0
hmod(  1, 7) =   1
hmod(  3, 7) =   3
hmod(  4, 7) =  -3
hmod(  7, 7) =   0
hmod(  8, 7) =   1

hmod( 50, 101) =  50
hmod( 51, 101) = -50
hmod( 60, 101) = -41
hmod(100, 101) =  -1
hmod(101, 101) =   0


In [4]:
# The helper matches the library's coefficient-wise reduce() exactly.
alg_raw = Algebra(3, 0, dtype="int64")   # no modulus: coefficients stay raw

for v in (-60, -4, 0, 50, 51, 60, 100, 150):
    lib = alg_raw({1: v}).reduce(101)["e1"]
    assert hmod(v, 101) == lib, (v, hmod(v, 101), lib)

print("hmod(v, 101) == alg_raw({1: v}).reduce(101)['e1'] for all test values")

hmod(v, 101) == alg_raw({1: v}).reduce(101)['e1'] for all test values


## 4. Automatic reduction — the single-modulus algebra

With `modulus=101` stored on `alg`, every operation reduces its result
automatically. No `_mod` suffix is needed.

In [5]:
e1 = alg("e1")
e2 = alg("e2")

# Scalar multiplication wraps large coefficients.
(60 * e1).show("60 * e1   (hmod(60, 101) = -41)")
(50 * e1).show("50 * e1   (hmod(50, 101) =  50)")
(51 * e1).show("51 * e1   (hmod(51, 101) = -50)")

print()

# Addition reduces after the sum of same-blade coefficients.
a = 48 * e1 + 48 * e2
b = 10 * e1 + 40 * e2
a.show("a = 48 e1 + 48 e2")
b.show("b = 10 e1 + 40 e2")
(a + b).show("a + b      (e1: hmod(58,101)=-43, e2: hmod(88,101)=-13)")

60 * e1   (hmod(60, 101) = -41): -41 e1
50 * e1   (hmod(50, 101) =  50): 50 e1
51 * e1   (hmod(51, 101) = -50): -50 e1

a = 48 e1 + 48 e2             : 48 e1 + 48 e2
b = 10 e1 + 40 e2             : 10 e1 + 40 e2
a + b      (e1: hmod(58,101)=-43, e2: hmod(88,101)=-13): -43 e1 - 13 e2


In [6]:
# Geometric products stay inside the ring.
x = alg("3 e1 + 7 e2 - 2 e3")
y = alg("5 e1 - 4 e2 + 6 e3")

x.show("x")
y.show("y")
(x * y).show("x * y")
(y * x).show("y * x")
(x * y + y * x).show("x*y + y*x   (= 2 (x|y), a scalar)")

x                             : 3 e1 + 7 e2 - 2 e3
y                             : 5 e1 - 4 e2 + 6 e3
x * y                         : -25 - 47 e12 + 28 e13 + 34 e23
y * x                         : -25 + 47 e12 - 28 e13 - 34 e23
x*y + y*x   (= 2 (x|y), a scalar): -50


## 5. Modular multiplicative inverse

For a single-modulus algebra, `inv()` uses the stored modulus automatically and
returns the inverse in the ring: `a * a.inv() == 1` (scalar part exactly 1).

In [7]:
a = alg("3 e1 + 7 e2 - 2 e3 + 4 e12")

a.show("a")
a_inv = a.inv()
a_inv.show("a.inv()")
(a * a_inv).show("a * a.inv()")
print("scalar coefficient:", (a * a_inv)["s"])

a                             : 3 e1 + 7 e2 - 2 e3 + 4 e12
a.inv()                       : 9 e1 + 21 e2 + 3 e3 - 34 e12 + 41 e13 - 32 e23
a * a.inv()                   : 1
scalar coefficient: 1


## 6. Multiple moduli on one algebra — the NTRU pattern

A single stored modulus cannot express an algorithm that needs the *same*
multivector reduced under *two different* moduli — the situation in NTRU-style
geometric-algebra cryptosystems. The solution is to create **one** algebra with
**no** modulus and use the explicit per-call methods:

| Method | Meaning |
|---|---|
| `a.gp_mod(b, p)` | geometric product, then `hmod(·, p)` |
| `a.op_mod(b, p)` | outer product, then `hmod(·, p)` |
| `a.ip_mod(b, p)` | inner product, then `hmod(·, p)` |
| `a.inv(p)` | modular inverse mod prime `p` |
| `a.reduce(p)` | standalone `hmod(·, p)` applied coefficient-wise |

In [8]:
alg2 = Algebra(3, 0, dtype="int64")   # no stored modulus

a = alg2("3 e1 + 2 e2")
b = alg2("e1 + 4 e12")

P = 5
Q = 41

a.show("a")
b.show("b")
a.gp_mod(b, P).show(f"a.gp_mod(b, P={P})")
a.gp_mod(b, Q).show(f"a.gp_mod(b, Q={Q})")

print()
raw = a * b
raw.show("a * b        (raw, no reduction)")
raw.reduce(P).show(f"(a*b).reduce(P={P})")
raw.reduce(Q).show(f"(a*b).reduce(Q={Q})")

a                             : 3 e1 + 2 e2
b                             : e1 + 4 e12
a.gp_mod(b, P=5)              : -2 + 2 e1 + 2 e2 - 2 e12
a.gp_mod(b, Q=41)             : 3 - 8 e1 + 12 e2 - 2 e12

a * b        (raw, no reduction): 3 - 8 e1 + 12 e2 - 2 e12
(a*b).reduce(P=5)             : -2 + 2 e1 + 2 e2 - 2 e12
(a*b).reduce(Q=41)            : 3 - 8 e1 + 12 e2 - 2 e12


## 7. NTRU-style encode / decode

As a concrete use of the two-modulus pattern, here is a miniature NTRU-style
round-trip, mirroring the structure of pytanga's `Tan.Crypt.Test` C++ experiments:

- **Alice's secret key** `f` must be invertible under both moduli.
- **Public key** `h = f_inv_B * g  (mod MODB)`.
- **Bob encodes** `c = h * (MODA * L) + M  (mod MODB)`.
- **Alice decodes** `S1 = f * c (mod MODB)`, then `M' = f_inv_A * S1 (mod MODA)`.

`G(3,0)` has `algebra_dim = 8`; the moduli follow the reference sizing
`MODA = next_prime(2*half_range+1)` and `MODB = next_prime(MODA*half_range²*algebra_dim)`.

In [9]:
import random

HALF_RANGE = 1
MODA = 5
MODB = 41

rng = random.Random(42)


def rand_mv():
    """Random MV with coefficients in [-HALF_RANGE, HALF_RANGE]."""
    return alg2({blade_id: rng.randint(-HALF_RANGE, HALF_RANGE)
                 for blade_id in alg2.all_blades()})


# Alice chooses a secret key f invertible under both moduli.
for _ in range(20):
    f = rand_mv()
    try:
        f_inv_A = f.inv(MODA)
        f_inv_B = f.inv(MODB)
        break
    except RuntimeError:
        continue
else:
    raise RuntimeError("could not find an invertible f in 20 tries")

g = rand_mv()
h = f_inv_B.gp_mod(g, MODB)              # public key

f.show("Alice's secret key f")
h.show("public key h = f_inv_B * g (mod 41)")

# Bob encodes a message.
L = rand_mv()
M = rand_mv()
c = (h * (MODA * L) + M).reduce(MODB)    # ciphertext

M.show("plaintext M")
c.show("ciphertext c")

# Alice decodes.
S1 = f.gp_mod(c, MODB)
M_decoded = f_inv_A.gp_mod(S1, MODA)

M_decoded.show("decoded M'")
diff = M.reduce(MODA) - M_decoded
print("M' == M (mod MODA):", all(v == 0 for v in diff.to_dict().values()))

Alice's secret key f          : e1 - e3 - e12 + e13
public key h = f_inv_B * g (mod 41): 1 - 2 e1 - 20 e2 - 20 e12 + 19 e13 - 20 I
plaintext M                   : -1 - e2 + e3 + e13 + e23
ciphertext c                  : 9 + 5 e1 + 9 e2 - 9 e3 - 15 e12 + e13 - 4 e23 + 10 I
decoded M'                    : -1 - e2 + e3 + e13 + e23
M' == M (mod MODA): True


## 8. The modular solver — `solve_mod`

`solve_mod(A, C, modulus=..., algebra=...)` solves `A * X = C` modulo a prime by
building the product matrix and running Gaussian elimination over `Z/pZ` (C++
`CCongruence_HMod`). Unlike the float solver, Gaussian elimination needs a
**square** system, so the automatically-derived blade masks for `A` and `C` must
have equal size — otherwise it raises `ValueError`.

A clean example is the modular inverse of a vector: solve `A * X = 1` with
`A = 3 e1 + 5 e2`. The unknown `X` lives in `{e1, e2}` and the result `A * X`
spans `{scalar, e12}` — two blades each, so the system is square.

In [10]:
alg3 = Algebra(3, 0, dtype="int64")      # no stored modulus

A = alg3("3 e1 + 5 e2")
X = solve_mod(A, 1, modulus=97, algebra=alg3)

A.show("A")
X.show("X = solve_mod(A, 1, modulus=97)")
(A.gp_mod(X, 97)).show("A * X (mod 97)")

A                             : 3 e1 + 5 e2
X = solve_mod(A, 1, modulus=97): -37 e1 + 3 e2
A * X (mod 97)                : 1


In [11]:
# If the modulus is stored on the algebra, it is used automatically ...
alg97 = Algebra(3, 0, dtype="int64", modulus=97)
A97 = alg97("3 e1 + 5 e2")

print("solve_mod(A, 1) :", solve_mod(A97, 1, algebra=alg97).to_dict())
print("A.inv()          :", A97.inv().to_dict())

solve_mod(A, 1) : {'e1': -37, 'e2': 3}
A.inv()          : {'e1': -37, 'e2': 3}


## 9. Use cases

- **Lattice-based cryptography.** NTRU-style key exchange and encryption need
  exact integer arithmetic reduced under two carefully-sized primes. `hmod` keeps
  coefficients centred (small magnitude), which is exactly what these schemes need
  to avoid wrap-around errors during decryption.
- **Exact symbolic computation.** Integer dtypes give exact coefficients — no
  floating-point residue to `prune()`. With a prime modulus you get deterministic,
  reproducible results and clean zero tests.
- **Field arithmetic.** For a prime modulus `p`, `Z/pZ` is a field, so every
  non-zero pivot has a multiplicative inverse — which is why `inv()` and
  `solve_mod` work. Non-prime moduli can still be used when every pivot is
  coprime to the modulus.

> **Limits:** `solve_mod` currently supports the geometric product only, and SVD /
> least-squares are not available over integer rings (they require floating-point
> division). Use the float `solve` / `solve_lsq` for those. See
> [Tutorial 11](../11_equation_solving/).

## 10. Summary & next steps

| Concept | API |
|---|---|
| Integer algebra | `Algebra(dim, sig, dtype="int64")` |
| Fixed modulus | `Algebra(dim, sig, dtype="int64", modulus=p)` |
| Half-space reduction | automatic on a fixed-modulus algebra; `a.reduce(p)` |
| Modular geometric product | `a.gp_mod(b, p)` |
| Modular outer / inner product | `a.op_mod(b, p)` / `a.ip_mod(b, p)` |
| Modular inverse | `a.inv(p)` (or `a.inv()` with a stored modulus) |
| Modular solver | `solve_mod(A, C, modulus=p, algebra=alg)` |

**Where to go next:**

- [**10 · BladeMask — Labelling Subspaces**](../10_blade_mask/) — the type that
  labels solver rows, tensor axes, and matrix dimensions.
- [**11 · Equation Solving**](../11_equation_solving/) — the full solver pipeline,
  including `solve`, `solve_lsq`, and `solve_mod`.